# 05_illumination_benchmark.ipynb

Task 4.3. Đánh giá các kỹ thuật cải thiện ánh sáng, tương phản và màu: Gamma, Gamma tự động, CLAHE, Retinex và gray-world, trên tám họ suy giảm của benchmark 4.1. Model là reference model ConvNeXt-Tiny đóng băng, không train lại.

Chỉ dùng tập validation. Tập test không được mở, tham số chốt ở đây rồi mới sang task 4.5.

Notebook làm năm việc: kiểm tra ảnh suy giảm khớp với 4.1, tối ưu tham số từng kỹ thuật trên các họ phù hợp, chọn cấu hình, chạy đối chứng trên các họ không phù hợp, và đo chi phí trên ảnh sạch. Mỗi bước có biểu đồ để quan sát.

## 1. Môi trường

Internet để On. Accelerator chọn GPU (T4). Dataset input cần hai cái: `cthng123/durian-leaf-disease-dataset` và dataset chứa checkpoint `convnext_tiny_seed42.pt` (cùng dataset đã dùng cho notebook 03 và 04).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Đổi nhánh tại đây, rồi chạy lại cell.
GIT_BRANCH = "illumination-benchmark"
REPO_URL = "https://github.com/meo225/durian-leaf-disease-robustness.git"
REPO_DIR = Path("/kaggle/working") / "durian-leaf-disease-robustness"

if not (Path("/kaggle/working").is_dir() and Path("/kaggle/input").is_dir()):
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle. Xem README, mục Chạy trên Kaggle.")

def git(args, cwd=None):
    subprocess.run(
        ["git", *args],
        cwd=cwd,
        check=True,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )


if not (REPO_DIR / ".git").is_dir():
    git(["clone", "--branch", GIT_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    git(["fetch", "origin", GIT_BRANCH], cwd=REPO_DIR)
    git(["checkout", GIT_BRANCH], cwd=REPO_DIR)
    git(["reset", "--hard", f"origin/{GIT_BRANCH}"], cwd=REPO_DIR)

matches = []
seen = set()
for path in Path("/kaggle/input").rglob("DLD_FinalDataset_224_spit"):
    resolved = str(path.resolve())
    if path.is_dir() and resolved not in seen:
        seen.add(resolved)
        matches.append(path)

if not matches:
    found = []
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for path in sorted(input_root.rglob("*")):
            if path.is_dir() and len(path.relative_to(input_root).parts) <= 3:
                found.append(str(path.relative_to(input_root)))
    listing = "\n".join(found[:40]) if found else "(trống)"
    raise FileNotFoundError(
        "Chưa thấy DLD_FinalDataset_224_spit trong /kaggle/input.\n"
        "Add Input: cthng123/durian-leaf-disease-dataset, rồi Save & Run All.\n"
        f"Thư mục hiện có:\n{listing}"
    )

image_root = matches[0].parent
link = REPO_DIR / "datasets" / "durian-ldd"
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink() or link.is_file():
    link.unlink()
elif link.exists():
    link.rmdir()
link.symlink_to(image_root, target_is_directory=True)

if not (link / "DLD_FinalDataset_224_spit").is_dir():
    raise FileNotFoundError(f"Không thấy ảnh tại {link}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True
).strip()
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, text=True
).strip()
print(f"Nhánh: {branch}")
print(f"Commit: {commit}")
print(f"Ảnh: {link / 'DLD_FinalDataset_224_spit'}")

## 2. Checkpoint và dữ liệu

Nạp reference model, tạo 24 điều kiện suy giảm của scope task này trên 444 ảnh validation, lưu dạng uint8. Mọi cấu hình xử lý chạy lên chính các ảnh này.

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import Image, display
from torch import nn

from src.corruption import build_loader, load_conditions, load_seed
from src.evaluation import plot_corruption_examples
from src.evaluation.processing_benchmark import (
    ImageCache, Runner, best_per_condition, build_matrix, clean_cost_table, example_grid, load_grid,
    make_processor, run_clean_cost, run_controls, run_tuning, save_predictions, scope_conditions, select_configs,
)
from src.evaluation.processing_plots import (
    plot_best_per_condition, plot_clean_cost, plot_confusion_cases, plot_grid_search, plot_luminance_histograms,
    plot_matrix_heatmap, plot_rank_table,
)
from src.evaluation.reference import collect_checkpoint, load_trained_model
from src.processing.illumination import mean_luminance
from src.training.loop import evaluate_loader, resolve_device

FIG = REPO_DIR / "outputs/figures/illumination"
RES = REPO_DIR / "outputs/results/illumination"
REP = REPO_DIR / "reports/processing"
for folder in (FIG, RES, REP):
    folder.mkdir(parents=True, exist_ok=True)


def show(path):
    display(Image(filename=str(path)))


grid_cfg = load_grid("configs/illumination.yaml")
pipeline_cfg = yaml.safe_load((REPO_DIR / "configs/pipeline.yaml").read_text(encoding="utf-8"))
classes = list(pipeline_cfg["classes"])
reference = json.loads((REPO_DIR / "reports/baselines/reference_selection.json").read_text(encoding="utf-8"))["selection"]["reference"]
assert reference == "convnext_tiny", f"Reference model là {reference}, notebook này viết cho convnext_tiny."
train_cfg = yaml.safe_load((REPO_DIR / "configs/train_convnext_tiny.yaml").read_text(encoding="utf-8"))

checkpoint = collect_checkpoint("/kaggle/input", grid_cfg["checkpoint_name"], "/kaggle/working/checkpoints")
device = resolve_device()
model = load_trained_model(train_cfg, checkpoint, len(classes), device)
print(f"Reference model: {reference}, thiết bị: {device}")

seed = load_seed(grid_cfg["corruption_config"])
conditions = scope_conditions(load_conditions(grid_cfg["corruption_config"], "val"), grid_cfg["scope_families"])
families = list(grid_cfg["scope_families"])
print(f"{len(conditions)} điều kiện thuộc {len(families)} họ: {', '.join(families)}")

start = time.time()
cache = ImageCache(pipeline_cfg, conditions, seed)
for name in ["clean", *conditions]:
    cache.get(name)
print(f"{len(cache)} ảnh validation, {len(conditions) + 1} bộ ảnh uint8, mất {time.time() - start:.0f} giây")

### Kiểm tra khớp với benchmark 4.1

Ảnh suy giảm ở đây được tạo một lần rồi dùng lại, khác đường đi của loader ở 4.1. Ô dưới so điểm "không xử lý" của ba điều kiện giữa hai đường đi. Hai số phải trùng nhau, nếu lệch thì dừng.

In [ ]:
runner = Runner(model, cache, pipeline_cfg, device, progress=print)
criterion = nn.CrossEntropyLoss()
for name in ["glare_s2", "shadow_s3", "color_cast_s1"]:
    expected = evaluate_loader(model, build_loader(pipeline_cfg, "val", name, conditions[name], seed), criterion, device, classes)["macro_f1"]
    runner.run("raw", "none", "none", None, name)
    mine = runner.rows[-1]["macro_f1"]
    print(f"{name:16s} loader 4.1: {expected:.6f}   cache: {mine:.6f}")
    assert abs(mine - expected) < 1e-6, "Ảnh suy giảm không khớp pipeline 4.1"
print("Khớp.")

## 3. Xem trước các kỹ thuật

Chưa tối ưu gì. Mỗi họ suy giảm (dòng) qua các kỹ thuật với một cấu hình mặc định (cột): ảnh sạch, ảnh suy giảm, rồi kết quả xử lý. Xem kỹ thuật nào sửa đúng loại hỏng, kỹ thuật nào làm ảnh xấu hơn hoặc mất vết bệnh.

In [ ]:
preview_choice = {"gamma": "gamma_0.6", "gamma_auto": "auto_0.5", "clahe": "clahe_c2_t8", "retinex": "msr_small_a1.0", "gray_world": "gw_1.0"}
preview = {
    technique: {"per_family": {}, "general": {"config_id": config, "params": grid_cfg["techniques"][technique]["grid"][config]}}
    for technique, config in preview_choice.items()
}
diseased = int(np.where(cache.labels == classes.index("LEAF_BLIGHT"))[0][0])
for severity, index in [(2, 0), (3, diseased)]:
    path = FIG / f"preview_s{severity}_img{index}.png"
    plot_corruption_examples(example_grid(cache, preview, index, severity, families), path, dpi=100)
    print(f"Ảnh {index} ({classes[int(cache.labels[index])]}), mức {severity}")
    show(path)

## 4. Điểm khi không xử lý

Macro F1 của reference model trên ảnh sạch và 24 điều kiện. Cột `lệch 4.1` so với bảng hiệu chỉnh ở task 4.1 (làm tròn 3 chữ số). Lệch dưới 0,001 là khớp.

In [ ]:
runner.raw()
frame = runner.frame()
raw = frame[frame["role"] == "raw"]
clean_f1 = float(raw[raw["condition"] == "clean"]["macro_f1"].iloc[0])
print(f"Macro F1 ảnh sạch: {clean_f1:.4f}")
raw_table = raw[raw["condition"] != "clean"].pivot(index="family", columns="severity", values="macro_f1").reindex(families)

recorded_41 = {
    "brightness_bright": [0.950, 0.804, 0.576], "brightness_dark": [0.954, 0.892, 0.604], "color_cast": [0.944, 0.889, 0.820],
    "contrast_high": [0.933, 0.713, 0.528], "contrast_low": [0.956, 0.874, 0.426], "glare": [0.968, 0.954, 0.918],
    "lowlight": [0.904, 0.826, 0.637], "shadow": [0.958, 0.908, 0.792],
}
diff = pd.DataFrame({family: np.round(raw_table.loc[family].values, 3) - np.array(values) for family, values in recorded_41.items()}, index=[1, 2, 3]).T
display(raw_table.round(3).assign(**{"lệch 4.1 lớn nhất": diff.abs().max(axis=1).round(4)}))
if diff.abs().max().max() > 0.002:
    print("CẢNH BÁO: có điểm lệch hơn 0,002 so với bảng 4.1 trong reports/corruptions/README.md.")

## 5. Tối ưu tham số trên validation

Mỗi kỹ thuật chạy toàn bộ lưới tham số trên các họ suy giảm phù hợp với nó (xem `configs/illumination.yaml`). Ô này chạy lâu nhất, khoảng 15 đến 25 phút trên T4. Log in mỗi cấu hình khi xong.

In [ ]:
start = time.time()
run_tuning(runner, grid_cfg)
print(f"Tối ưu xong sau {(time.time() - start) / 60:.1f} phút")
run_clean_cost(runner, grid_cfg)
frame = runner.frame()
frame.to_csv(RES / "illumination_grid.csv", index=False)
print(f"{len(frame)} dòng kết quả, đã lưu {RES / 'illumination_grid.csv'}")

### Kết quả lưới tham số

Mỗi ô nhỏ là một họ suy giảm. Trục ngang là cấu hình, ba cột màu là ba mức. Đường đứt cùng màu là điểm khi không xử lý, nên cột nào vượt đường đứt cùng màu là cấu hình đó giúp được. Đường chấm xám là điểm ảnh sạch, tức mức tối đa có thể phục hồi.

In [ ]:
for technique, spec in grid_cfg["techniques"].items():
    path = FIG / f"grid_{technique}.png"
    plot_grid_search(frame, technique, spec["families"], list(spec["grid"]), path)
    print(f"== {technique}")
    show(path)

## 6. Chọn cấu hình

Với mỗi cặp kỹ thuật và họ phù hợp, chọn cấu hình có macro F1 trung bình trên ba mức cao nhất. Nếu có cấu hình trong khoảng 0,003 so với cấu hình tốt nhất thì lấy cấu hình giữ F1 ảnh sạch cao hơn. Mỗi kỹ thuật còn một cấu hình chung (điểm trung bình cao nhất qua mọi họ phù hợp), dùng làm "kỹ thuật cố định" và cho đối chứng không phù hợp.

In [ ]:
selected = select_configs(frame, grid_cfg)
raw_mean = raw[raw["condition"] != "clean"].groupby("family")["macro_f1"].mean()
summary = pd.DataFrame(
    [
        {"kỹ thuật": technique, "họ": family, "cấu hình": item["config_id"], "F1 sau xử lý": item["mean_macro_f1"],
         "F1 không xử lý": raw_mean[family], "tăng": item["mean_macro_f1"] - raw_mean[family]}
        for technique, entry in selected.items() for family, item in entry["per_family"].items()
    ]
)
display(summary.round(4))
general = pd.DataFrame(
    [{"kỹ thuật": technique, "cấu hình chung": entry["general"]["config_id"], "F1 trung bình họ phù hợp": entry["general"]["mean_macro_f1"],
      "F1 ảnh sạch sau xử lý": entry["general"]["clean_macro_f1"], "F1 ảnh sạch gốc": clean_f1} for technique, entry in selected.items()]
)
display(general.round(4))

## 7. Đối chứng không phù hợp và bản đồ phục hồi

Cấu hình chung của mỗi kỹ thuật chạy thêm trên các họ không phù hợp với nó, để thấy kỹ thuật nào làm hại ở đâu. Rồi ghép thành bảng kỹ thuật × 24 điều kiện.

Chỉ số: `recovery` = F1 sau xử lý trừ F1 suy giảm. `normalized recovery` = recovery chia mức giảm (F1 sạch trừ F1 suy giảm), chỉ tính khi mức giảm lớn hơn 0,02. Gần 1 là lấy lại gần hết phần đã mất, 0 là không đổi, âm là xử lý làm tệ hơn.

In [ ]:
run_controls(runner, grid_cfg, selected)
frame = runner.frame()
matrix = build_matrix(frame, selected, float(grid_cfg["selection"]["min_degradation"]))
matrix.to_csv(RES / "illumination_matrix.csv", index=False)
print(f"{len(matrix)} ô trong bản đồ phục hồi ({matrix['suitable'].sum()} cặp phù hợp, {(~matrix['suitable']).sum()} cặp đối chứng)")

In [ ]:
plot_matrix_heatmap(matrix, "recovery", families, FIG / "recovery_heatmap.png", "Recovery trên validation (F1 sau xử lý trừ F1 suy giảm). Ô khung đen là cặp phù hợp.")
show(FIG / "recovery_heatmap.png")

Cách đọc: xanh là xử lý giúp, đỏ là làm hại. Ô có khung đen là cặp kỹ thuật và họ phù hợp, ô không khung là đối chứng. Một kỹ thuật tốt là xanh trong khung đen, và không đỏ đậm ngoài khung.

In [ ]:
plot_matrix_heatmap(matrix, "normalized_recovery", families, FIG / "normalized_recovery_heatmap.png", "Normalized recovery (1 là lấy lại hết phần đã mất). Ô trống là điều kiện mức giảm nhỏ hơn 0,02.", limit=1.0)
show(FIG / "normalized_recovery_heatmap.png")

In [ ]:
plot_rank_table(matrix, "recovery", FIG / "rank_recovery.png", "Recovery trung bình trên ba mức, ô xanh lá là kỹ thuật tốt nhất mỗi họ")
show(FIG / "rank_recovery.png")

## 8. Kỹ thuật tốt nhất mỗi điều kiện

Với mỗi điều kiện, kỹ thuật cho macro F1 cao nhất trên validation (kể cả cặp không phù hợp). Đây chỉ để quan sát. Kết luận sạch phải đo lại trên test ở task 4.5, vì chọn kỹ thuật tốt nhất trên cùng tập đo sẽ lạc quan.

In [ ]:
best = best_per_condition(matrix)
best.to_csv(RES / "illumination_best_per_condition.csv")
display(best.round(3))
plot_best_per_condition(best, clean_f1, FIG / "best_per_condition.png")
show(FIG / "best_per_condition.png")

### Ảnh minh họa với cấu hình đã chọn

Cùng bố cục với mục 3 nhưng dùng cấu hình đã chọn cho từng họ. Cột sau cột "suy giảm" là các kỹ thuật theo thứ tự gamma, gamma_auto, clahe, retinex, gray_world.

In [ ]:
for severity, index in [(2, 0), (3, diseased), (3, int(np.where(cache.labels == classes.index("HEALTHY_LEAF"))[0][0]))]:
    path = FIG / f"selected_s{severity}_img{index}.png"
    plot_corruption_examples(example_grid(cache, selected, index, severity, families), path, dpi=100)
    print(f"Ảnh {index} ({classes[int(cache.labels[index])]}), mức {severity}")
    show(path)

## 9. Chi phí trên ảnh sạch

Khi ảnh không hỏng mà vẫn bị xử lý thì điểm đổi bao nhiêu. Bên trái là thay đổi macro F1 tổng, bên phải là thay đổi recall từng lớp. Một kỹ thuật làm giảm mạnh một lớp bệnh là dấu hiệu xử lý xóa mất đặc trưng của lớp đó.

In [ ]:
cost = clean_cost_table(frame, selected)
cost.to_csv(RES / "illumination_clean_cost.csv", index=False)
display(cost[["technique", "config_id", "macro_f1", "change_macro_f1"]].round(4))
plot_clean_cost(cost, classes, FIG / "clean_cost.png")
show(FIG / "clean_cost.png")

## 10. Ca đáng chú ý

Confusion matrix của hai điều kiện được phục hồi nhiều nhất và hai điều kiện bị làm tệ nhất. Mỗi cặp: trái là không xử lý, phải là sau xử lý.

In [ ]:
from sklearn.metrics import confusion_matrix

def matrix_of(technique, config_id, condition):
    preds = runner.predictions[f"{technique}|{config_id}|{condition}"].astype(int)
    return confusion_matrix(cache.labels, preds, labels=list(range(len(classes)))).tolist()

picked = pd.concat([matrix.nlargest(2, "recovery"), matrix.nsmallest(2, "recovery")])
cases = []
for row in picked.itertuples():
    tag = "giúp" if row.recovery > 0 else "hại"
    cases.append((f"{row.condition} không xử lý, F1 {row.raw_macro_f1:.3f}", matrix_of("none", "none", row.condition)))
    cases.append((f"{row.condition} sau {row.technique} {row.config_id} ({tag}), F1 {row.macro_f1:.3f}", matrix_of(row.technique, row.config_id, row.condition)))
plot_confusion_cases(cases, classes, FIG / "confusion_cases.png")
show(FIG / "confusion_cases.png")

## 11. Phân bố độ sáng trước và sau xử lý

Với bốn điều kiện mức 3, histogram độ sáng trung bình mỗi ảnh ở ba trạng thái. Xử lý tốt thì cột xanh lá (đã xử lý) dời về gần cột xanh dương (sạch).

In [ ]:
items = []
for condition in ["brightness_dark_s3", "brightness_bright_s3", "lowlight_s3", "contrast_low_s3"]:
    entry = best.loc[condition]
    processor = make_processor(entry["technique"], dict(grid_cfg["techniques"][entry["technique"]]["grid"][entry["config_id"]]))
    degraded = cache.get(condition).float() / 255.0
    items.append((f"{condition} sau {entry['technique']}", {
        "sạch": [mean_luminance(image) for image in cache.get("clean").float() / 255.0],
        "suy giảm": [mean_luminance(image) for image in degraded],
        "đã xử lý": [mean_luminance(processor(image)) for image in degraded],
    }))
plot_luminance_histograms(items, FIG / "luminance_histograms.png")
show(FIG / "luminance_histograms.png")

## 12. Lưu kết quả

Ghi file lưới tham số, bản đồ phục hồi, dự đoán từng ảnh (cho kiểm định ghép cặp ở 4.5) và cấu hình đã chọn. File cấu hình đã chọn nằm ở `reports/processing/illumination_selected.json`, đây là file task 4.5 và Phase 5 đọc.

In [ ]:
import shutil
import zipfile

save_predictions(runner, RES / "illumination_predictions.npz")
record = {
    "task": "4.3",
    "split": "val",
    "reference_model": reference,
    "corruption_config": grid_cfg["corruption_config"],
    "grid_config_version": grid_cfg["version"],
    "clean_macro_f1": clean_f1,
    "selection_rule": grid_cfg["selection"],
    "selected": selected,
    "clean_cost": cost.to_dict(orient="records"),
    "note": "Tham số chọn trên validation. Tập test chưa mở. Task 4.5 đo bản đồ phục hồi trên test với đúng các cấu hình này.",
}
(REP / "illumination_selected.json").write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
for name in ["recovery_heatmap", "normalized_recovery_heatmap", "rank_recovery", "best_per_condition", "clean_cost", "confusion_cases", "luminance_histograms"]:
    shutil.copy(FIG / f"{name}.png", REP / f"illumination_{name}.png")
for path in FIG.glob("selected_s*.png"):
    shutil.copy(path, REP / f"illumination_{path.name}")
for name in ["illumination_matrix.csv", "illumination_best_per_condition.csv", "illumination_clean_cost.csv"]:
    shutil.copy(RES / name, REP / name)

archive = Path("/kaggle/working/illumination_outputs.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for folder in (RES, FIG, REP):
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                bundle.write(path, path.relative_to(REPO_DIR).as_posix())
print(f"Đã nén {archive} ({archive.stat().st_size / 1024**2:.1f} MB)")
print()
print("Quan sát trước: bản đồ phục hồi (recovery_heatmap.png), chi phí ảnh sạch (clean_cost.png), file illumination_selected.json.")